# Air Passengers EDA

World Bank air transport passenger data (1960–2023). Source: World Bank Open Data: IS.AIR.PSGR.

## 1. Setup

In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, name: str) -> None:
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


In [ ]:
BLUE   = "#2176AE"
ORANGE = "#E76F51"
TEAL   = "#2A9D8F"
SLATE  = "#264653"
RED    = "#E63946"
GRAY   = "#C8C8C8"

plt.rcParams.update({
    "figure.dpi":          130,
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           False,
    "font.family":         "sans-serif",
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.frameon":      False,
    "figure.titlesize":    13,
    "figure.titleweight":  "bold",
})

## 2. Load Clean Data

In [ ]:
df = pd.read_csv(CLEAN_DIR / "passenger_clean.csv")
print(f"Shape: {df.shape}")
df.info()
df.head()


## 3. Exploratory Data Analysis

In [ ]:
year_cols = [c for c in df.columns if str(c).isdigit()]

df_long = df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Passengers",
)
df_long["Year"]       = df_long["Year"].astype(int)
df_long["Passengers"] = pd.to_numeric(df_long["Passengers"], errors="coerce")
df_long = df_long[df_long["Year"] <= 2023]

REGIONAL_CODES = {
    "WLD","EAS","ECS","LCN","MEA","NAC","SAS","SSF","EAP","ECA","LAC","MNA",
    "SSA","HIC","MIC","LMC","UMC","LIC","IBRD","IDA","OED","AFE","AFW","ARB",
    "CEB","CSS","EAR","EMU","EUU","FCS","HPC","IBD","IBT","IDX","IDB","INX",
    "LDC","LTE","NAF","OSS","PRE","PSS","PST","SST","TEA","TEC","TLA","TMN",
    "TSA","TSS","XZN",
}
df_countries = df_long[~df_long["Country Code"].isin(REGIONAL_CODES)].copy()
df_regions   = df_long[ df_long["Country Code"].isin(REGIONAL_CODES)].copy()
print(f"Country rows: {len(df_countries):,}  |  Regional aggregate rows: {len(df_regions):,}")

### 3.1 Passenger Distribution (2019)

In [ ]:
data_2019 = df_countries[
    (df_countries["Year"] == 2019) & df_countries["Passengers"].notna()
].copy()

top20 = data_2019.nlargest(20, "Passengers")

bar_colors = [
    RED if c == "CHN" else
    BLUE if c == "USA" else
    SLATE
    for c in top20["Country Code"]
]

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    top20["Country Name"],
    top20["Passengers"] / 1e9,
    color=bar_colors,
    edgecolor="white"
)

ax.set_xlabel("Passengers (billions)")
ax.set_title("Top 20 countries by passengers (2019)")

ax.invert_yaxis()

ax.xaxis.set_major_formatter(
    mticker.FormatStrFormatter("%.1f B")
)

plt.tight_layout()
save_figure(fig, "passengers_top20_2019")
plt.show()

### 3.2 Global Air Passengers Trend (1970\u20132023)

In [ ]:
world_trend = df_long[df_long["Country Code"] == "WLD"].dropna(subset=["Passengers"])

if len(world_trend) < 5:
    world_trend = (
        df_countries.groupby("Year")["Passengers"]
        .sum()
        .reset_index(name="Passengers")
    )

world_trend = world_trend[world_trend["Year"] >= 1970]

events = {
    1973: ("Oil crisis", -0.3),
    1991: ("Gulf War",  -0.3),
    2001: ("9/11",      -0.3),
    2003: ("SARS",      -0.3),
    2009: ("GFC",       -0.3),
}

fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(
    world_trend["Year"],
    world_trend["Passengers"] / 1e9,
    alpha=0.12,
    color=BLUE
)

ax.plot(
    world_trend["Year"],
    world_trend["Passengers"] / 1e9,
    color=BLUE,
    lw=2.5
)

# Standard event annotations
for yr, (label, offset) in events.items():
    row = world_trend[world_trend["Year"] == yr]

    if len(row):
        y = float(row["Passengers"].values[0]) / 1e9

        ax.axvline(
            yr,
            color=RED,
            ls=":",
            lw=1,
            alpha=0.6
        )

        ax.annotate(
            label,
            xy=(yr, y),
            xytext=(yr + 0.3, y + offset),
            fontsize=7.5,
            color=RED,
            arrowprops=dict(
                arrowstyle="-",
                color=RED,
                lw=0.7
            )
        )

# Improved COVID annotation
covid_year = 2020
covid_row = world_trend[world_trend["Year"] == covid_year]

if len(covid_row):
    covid_y = float(covid_row["Passengers"].values[0]) / 1e9

    ax.axvline(
        covid_year,
        color=RED,
        ls=":",
        lw=1,
        alpha=0.6
    )

    ax.annotate(
        "COVID-19",
        xy=(covid_year, covid_y),
        xytext=(2020.3, covid_y + 0.5),
        fontsize=8.5,
        color=RED,
        fontweight="bold",
        ha="left",
        va="bottom",
        arrowprops=dict(
            arrowstyle="->",
            color=RED,
            lw=1
        ),
        bbox=dict(
            boxstyle="round,pad=0.25",
            fc="white",
            ec="none",
            alpha=0.8
        )
    )

ax.set_title("Global air passengers trend (1970–2023)")
ax.set_xlabel("Year")
ax.set_ylabel("Passengers (billions)")

ax.yaxis.set_major_formatter(
    mticker.FormatStrFormatter("%.1f B")
)

plt.tight_layout()
save_figure(fig, "passengers_global_trend")
plt.show()